# Anatomical Pipeline Walkthrough

### What this pipeline produces
Starting from a raw T1-weighted MRI, we end up with:
- A FreeSurfer reconstruction (cortical surface, grey/white matter boundaries)
- fmriprep anatomical processing
- A Benson14 atlas parcellating visual areas on that surface
- (Optional) Flatmaps via autoflatten + pycortex

### Prerequisites
Before starting, make sure you have completed the setup described in [config/local_setup.md](../config/local_setup.md). In brief you need:
- `PIPELINE_DIR` pointing to this repository in your `~/.bash_profile`
- Docker or Apptainer/Singularity installed
- mamba/conda installed
- FreeSurfer + freeview installed locally
- A `config/project_<name>.sh` file with your project paths
---

## 0. Set your variables

Fill in the cell below then **run it once**. It sets `PROJECT`, `SUB`, and `SES` as environment variables that every subsequent bash cell will inherit automatically — no need to re-source anything.

In [1]:
import subprocess, os
from cvl_utils.preproc_func import set_project
# ── Fill these in ────────────────────────────────────────────────────────────
PROJECT = 'lhon'    # matches your config/project_<name>.sh file
SUB     = 'sub-C001'  # subject label, e.g. sub-01
SES     = 'ses-prf'  # session containing the T1w scan, e.g. ses-01
# ─────────────────────────────────────────────────────────────────────────────
# Source the project config once and capture every exported variable into
# os.environ so all %%bash cells below pick them up without re-sourcing.
set_project(PROJECT)
# SUB/SES are notebook-only values (not part of the bash project config), so
# they must be exported explicitly or os.environ['SUB']/['SES'] won't exist.
os.environ['SUB'] = SUB
os.environ['SES'] = SES
!echo $SUB

Project:     lhon
BIDS dir:    /Users/marcusdaghlian/projects/lhon_sl
FreeSurfer:  /Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer
CTX dir:     (not set)
sub-C001


---
## Step 1 — fMRIPrep (anatomy only)

### What is fMRIPrep?
[fMRIPrep](https://fmriprep.org) is a robust preprocessing pipeline for fMRI data. Here we run it in **anatomy-only** mode, which does two things:
1. Runs **FreeSurfer** to reconstruct the cortical surface from the T1w image. FreeSurfer segments the brain into grey matter, white matter, and CSF, then builds explicit 3D mesh representations of the pial and white-matter boundaries. This typically takes 8–12 hours per subject.
2. Produces various anatomical derivatives (brain mask, tissue probability maps, MNI registration) that are used later in functional preprocessing.

### Why a separate FPREP_BIDS folder?
The script creates a stripped-down copy of the BIDS dataset inside `derivatives/FPREP_BIDS` containing only the T1w scan. We feed *this* to fMRIPrep rather than the full dataset. The reason: later in the pipeline we want to do some preprocessing steps ourselves (e.g. custom denoising), and we do not want fMRIPrep to pick up those files or get confused by them. Keeping the two cleanly separate makes the division between fMRIPrep's work and our own work explicit.

### Running locally vs. on the HPC
FreeSurfer is computationally intensive — on a modern laptop it takes around 8–12 hours per subject. If you have access to a High Performance Computing cluster you should run it there. The two options are shown below.

> **Note:** fMRIPrep runs inside a container (Docker or Apptainer/Singularity). Make sure your container manager is running before executing the cell.

In [4]:
%%bash 
# ── Option A: Run locally ─────────────────────────────────────────────────────
# Runs fMRIPrep + FreeSurfer on your local machine.
# Expect this to take 8-12 hours. Leave it running overnight.
${PIPELINE_DIR}/anatomical/s01_fmriprep_anat_only.sh --bids-dir $BIDS_DIR \
    --sub $SUB --ses $SES --suffix _test 

-------------------------------------------------------
Running fmriprep - for anatomy 
-------------------------------------------------------
 BIDS DIR:    /Users/marcusdaghlian/projects/lhon_sl
 Subject:   sub-C001
 Session:   ses-prf
-------------------------------------------------------
Copying anatomy
running fprep in /Users/marcusdaghlian/projects/lhon_sl/derivatives/FPREP_BIDS_test and fmriprep-25.2.4.sif


You are using fMRIPrep-25.2.4, and a newer version of fMRIPrep is available: 25.2.5.
Please check out our documentation about how and when to upgrade:
https://fmriprep.readthedocs.io/en/latest/faq.html#upgrading


260622-07:37:48,950 nipype.workflow IMPORTANT:
	 Running fMRIPrep version 25.2.4

         License NOTICE ##################################################
         fMRIPrep 25.2.4
         Copyright The NiPreps Developers.
         
         This product includes software developed by
         the NiPreps Community (https://nipreps.org/).
         
         Portions of this software were developed at the Department of
         Psychology at Stanford University, Stanford, CA, US.
         
         This software is also distributed as a Docker container image.
         The bootstrapping file for the image ("Dockerfile") is licensed
         under the MIT License.
         
         This software may be distributed through an add-on package called
         "Docker Wrapper" that is under the BSD 3-clause License.
         #################################################################
260622-07:37:48,951 nipype.workflow IMPORTANT:
	 Building fMRIPrep's workflow:
           * BIDS data

TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

---
## QC — Check the FreeSurfer surfaces

FreeSurfer's automatic segmentation is very good, but not perfect. Before proceeding you **must** check the surfaces by eye. Common problems to look for:

- **Sagittal sinus inclusion** — the large vein running along the midline can be misclassified as grey matter, pulling the pial surface inward and distorting the flat maps.
- **Dura mater inclusion** — similar issue, dura wrapping around the brain can be included as cortex.
- **Pial surface holes** — particularly around sulci, the pial surface can fail to close properly.
- **White matter over-erosion** — thin, convoluted gyri sometimes lose white matter vertices, making the surface look bumpy.

The blue lines = **pial** surface (outer boundary of grey matter). The yellow lines = **white** surface (grey/white boundary). They should hug the cortex tightly without wandering into ventricles, dura, or the sagittal sinus.

If you spot problems, you will need to manually correct the segmentation in freeview and re-run FreeSurfer (see the [FreeSurfer recon-all documentation](https://surfer.nmr.mgh.harvard.edu/fswiki/ReconAllDevTable) for how to do partial re-runs after edits).

In [5]:
%%bash
# Opens freeview with:
#   - T1 + brainmask + rawavg volumes
#   - lh/rh pial surfaces (blue)
#   - lh/rh white surfaces (yellow)
bash "${PIPELINE_DIR}/anatomical/qc_surfaces.sh" "${SUB}"

Opening subject sub-C001 in freeview...


### Optional: make a sagittal sweep movie

A movie sweeping through sagittal slices is very useful for quickly spotting sagittal sinus errors — they stand out clearly as the pial surface dipping inward when you can see the whole brain in motion. The movie is saved to `$SUBJECTS_DIR/<sub>/movie/<sub>.mp4`.

In [ ]:
%%bash
# Screenshots every sagittal slice (90-240 by default) then stitches into mp4
# Requires ffmpeg to be installed (brew install ffmpeg / apt install ffmpeg)
echo python "${PIPELINE_DIR}/anatomical/qc_surfaces_movie.py" "${SUB}"

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


Error: Subject directory not found for sub- in /Users/marcusdaghlian/projects/dp-clean-link/LHON_CF/data/bids/derivatives/freesurfer


CalledProcessError: Command 'b'# Screenshots every sagittal slice (90-240 by default) then stitches into mp4\n# Requires ffmpeg to be installed (brew install ffmpeg / apt install ffmpeg)\npython "${PIPELINE_DIR}/anatomical/qc_surfaces_movie.py" "${SUB}"\n'' returned non-zero exit status 1.

---
## Step 2 — Benson14 Visual Area Atlas

### What is the Benson14 atlas?
The Benson14 atlas ([Benson et al., 2014](https://doi.org/10.1371/journal.pcbi.1003538)) uses a template of retinotopic organisation to predict the locations of visual areas (V1, V2, V3, hV4, VO1/2, LO1/2, TO1/2, V3a/b) on an *individual's* cortical surface. It works by deforming a population-level template onto the subject's FreeSurfer surface using [neuropythy](https://github.com/noahbenson/neuropythy).

### What does this script do?
The script:
1. Calls `neuropythy atlas` to generate `.mgz` files (`lh/rh.benson14_varea.mgz`) in the subject's `surf/` folder, if they are not already there.
2. Reads those files and converts each area into a standard FreeSurfer `.label` file, saved in `label/custom/`.
3. Creates a combined `b14_V1V2V3.label` (the three primary visual areas together) and a `b14_ALL.label` containing all labelled visual cortex.

These label files can be loaded in freeview, used to mask functional data, or imported into pycortex (Step 4).

> **Environment:** This step requires the `b14` conda environment (neuropythy + nibabel). If you have not created it yet, run `bash ${PIPELINE_DIR}/config/s00_python_environments.sh --env b14`.

In [17]:
%%bash

conda run -n b14 python "${PIPELINE_DIR}/anatomical/s02_b14atlas.py" \
    --sub "${SUB}" \
    --fsdir "${SUBJECTS_DIR}" #--overwrite

Processing sub-C001
  Skipping lh (labels already exist, use --overwrite to regenerate)
  Skipping rh (labels already exist, use --overwrite to regenerate)
Done!



### Check what was created

In [18]:
%%bash
echo "=== Benson14 .mgz files (in surf/) ==="
ls "${SUBJECTS_DIR}/${SUB}/surf/"*benson14* 2>/dev/null || echo "  none found"

echo ""
echo "=== Label files (in label/custom/) ==="
ls "${SUBJECTS_DIR}/${SUB}/label/custom/" 2>/dev/null || echo "  none found"

=== Benson14 .mgz files (in surf/) ===


/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_angle.mgh
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_angle.mgz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_eccen.mgh
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_eccen.mgz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_sigma.mgh
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_sigma.mgz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_varea.mgh
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/lh.benson14_varea.mgz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/rh.benson14_angle.mgh
/Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer/sub-C001/surf/rh.benson14_angle.mgz
/Users/marcusdaghlia

---
## Summary: Output Files

After completing the pipeline your directory structure will look something like this:

```
$BIDS_DIR/
├── sub-01/ses-01/anat/          ← raw T1w (unchanged)
└── derivatives/
    ├── FPREP_BIDS/              ← stripped BIDS input fed to fMRIPrep
    ├── fmriprep/                ← fMRIPrep outputs (brain mask, MNI space etc.)
    ├── freesurfer/
    │   └── sub-01/
    │       ├── mri/             ← T1, brainmask, segmentation volumes
    │       ├── surf/            ← lh/rh.white, pial, inflated, benson14_varea.mgz
    │       ├── label/custom/    ← b14_V1.label, b14_V2.label, ... b14_ALL.label
    │       └── movie/           ← QC sagittal sweep mp4 (if you ran it)
```

You are now ready to move on to functional preprocessing.